# Checkpoint — Herhalingslabo Classificatie

Dit notebook wordt automatisch gecheckt via pytest (GitHub Actions). De checks voeren je notebook uit en valideren of bepaalde variabelen bestaan en binnen de verwachte ranges vallen. **Je krijgt geen hints in de foutmeldingen: het is aan jou om correct werkende code te schrijven.**

## BELANGRIJK: gebruik exact de genoemde variabelennamen

De checks zoeken volgende variabelen in je notebook. Andere namen worden niet gezien:

- `y_pred_diabetes_svc` — voorspellingen van het SVC-model op de diabetes-testset
- `accuracy_diabetes_svc` — accuracy van het SVC-model (float)
- `y_pred_diabetes_knn` — voorspellingen van het KNN-model op de diabetes-testset
- `accuracy_diabetes_knn` — accuracy van het KNN-model (float)
- `y_pred_churn` — voorspellingen van het churn-model op de testset
- `accuracy_churn` — accuracy van het churn-model (float)
- `y_pred_creditcard_ada` — voorspellingen van het AdaBoost-model
- `accuracy_creditcard_ada` — accuracy van het AdaBoost-model (float)
- `y_pred_creditcard_xgb` — voorspellingen van het XGBoost-model
- `accuracy_creditcard_xgb` — accuracy van het XGBoost-model (float)
- `antwoord_algoritme` — meerkeuzevraag 1: waarde `"A"`, `"B"`, `"C"` of `"D"`
- `antwoord_ongebalanceerd` — meerkeuzevraag 2: waarde `"A"`, `"B"`, `"C"` of `"D"`

Gebruik overal `test_size=0.2` en `random_state=42` bij de splits, zodat je resultaten reproduceerbaar zijn en binnen de verwachte drempels vallen.

Hoe weet je wanneer je welk algoritme best gebruikt ? Gebruik volgende handleiding:

**KNearestNeighbours**
- Geschikt voor classificatie en regressie
- Voor kleine/middelgrote datasets
- heel simpel, kan snel worden getrained
- werkt niet goed voor datasets met heel veel features

**Support Vector Machines**
- Vooral een standaard voor binaire classificatie, of wanneer je een duidelijke marge wilt op classificatie (=het hoofddoel van het algoritme)
- Voor kleine/middelgrote datasets
- Is kostelijker om te trainen dan KNN, maar classificeert dan wel sneller
- Kernel keuze en hyperparameter tuning zijn essentieel

**Decision Trees**
- Wanneer je een 'white box', 'explainable' model wilt
- kan zowel numerieke als categorische features makkelijk aan
- kan heel goed om met missende datapunten
- het risico op overfitting is groot. Bij twijfel, gebruik een random forest

**Random forest**
- Zelfde voordelen als decision tree, minder risico op overfitting
- Geschikt voor middelgrote/grote datasets
- Geschikt voor data met (heel) veel features
- Hebben uiteraard meer trainingstijd nodig dan een enkele decision tree

**Adaboost**
- Als data niet goed is gebalanceerd (niet evenveel items van elke klasse)
- Vermijd een onderfit. Dit model is beter dan 1 enkele onderliggende decision tree

**XGBoost**
- XGBoost is bijna *altijd* een goede keuze
- Middelgrote / grote datasets
- Datasets met (heel) veel features
- De implementatie van XGBoost staat, ondanks dat het een boosting algoritme is, toch deels parallellisatie toe.

In de praktijk zijn er vaak meerdere goede keuzes. Het is belangrijk te weten waarom een bepaald model wel/niet een goede keuze zou zijn, en vervolgens de meest veelbelovende modellen te proberen en te kijken wat het beste werkt.

## Oefening 1 — Diabetes

De dataset `data/diabetes.csv` bevat medische metingen; in de kolom `Outcome` staat of iemand diabetes heeft (1) of niet (0). Dit is een **kleine dataset** (768 rijen, 8 features).

Train twee modellen en vergelijk:
1. Een `SVC` met `kernel='linear'`
2. Een `KNeighborsClassifier` met `n_neighbors=5`

Gebruik `test_size=0.2` en `random_state=42`. Bewaar de voorspellingen in `y_pred_diabetes_svc` / `y_pred_diabetes_knn` en de accuracies in `accuracy_diabetes_svc` / `accuracy_diabetes_knn`.

In [11]:
# Libraries inladen en data klaarzetten
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

diabetes = pd.read_csv('data/diabetes.csv')

X = diabetes.drop(columns=['Outcome'])
y = diabetes['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Dataset onderzoeken
# print(diabetes.head())
# print(diabetes.tail())
# print(diabetes.info())
# print(y.value_counts(normalize=True)*100)
# De bovenstaande regels (in commentaar gezet voor overzicht) leren mij dat alle kolommen numeriek zijn en dat er geen ontbrekende waarden zijn. Er is enkel scaling nodig. De klassen zijn echter wel ongebalanceerd. Bij SVC kan dit verholpen worden door een gewicht toe te kennen bij de initialisatie van het model, bij KNN kan dit niet.

# Pipelines opzetten
pipeline_svc = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', SVC(random_state=42, kernel='linear'))
])
pipeline_knn = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

# Parametergrids voor GridsearchCV bepalen
param_grid_svc = {
    'model__C': [0.01,0.1,1,10,100,1000]
}
param_grid_knn = {
    'model__weights': ['distance', 'uniform'],
    'model__p': [1,2]
}

# Optimalisatie met gridsearchCV
grid_svc = GridSearchCV(pipeline_svc, param_grid_svc, cv=5, scoring='accuracy', n_jobs=-1)
grid_knn = GridSearchCV(pipeline_knn, param_grid_knn, cv=5, scoring='accuracy', n_jobs=-1)
grid_svc = grid_svc.fit(X_train, y_train)
grid_knn = grid_knn.fit(X_train, y_train)

# Evaluatie
best_svc = grid_svc.best_estimator_
best_knn = grid_knn.best_estimator_
y_pred_svc = best_svc.predict(X_test)
y_pred_knn = best_knn.predict(X_test)
print("SVC:")
print(classification_report(y_test, y_pred_svc))
print(confusion_matrix(y_test, y_pred_svc))
print()
print("KNN:")
print(classification_report(y_test, y_pred_knn))
print(confusion_matrix(y_test, y_pred_knn))
# Conclusie: Het balanceren van de klassen levert geen goede resultaten op bij SVC. Door dit niet te doen, worden de prestaties beter. De confusion matrices leren wel dat het aantal vals negatieven bij KNN hoger ligt. Op basis daarvan zou in deze medische context beargumenteerd kunnen worden dat SVC hier de betere keuze is (heeft ook een iets hogere accuracy). 



SVC:
              precision    recall  f1-score   support

           0       0.81      0.82      0.81        99
           1       0.67      0.65      0.66        55

    accuracy                           0.76       154
   macro avg       0.74      0.74      0.74       154
weighted avg       0.76      0.76      0.76       154

[[81 18]
 [19 36]]

KNN:
              precision    recall  f1-score   support

           0       0.75      0.80      0.77        99
           1       0.58      0.51      0.54        55

    accuracy                           0.69       154
   macro avg       0.66      0.65      0.66       154
weighted avg       0.69      0.69      0.69       154

[[79 20]
 [27 28]]


In [ ]:
# Train een SVC met kernel='linear' op de trainingsdata
# ...

# Maak voorspellingen op de testdata en bewaar ze in y_pred_diabetes_svc
# ...

# Bereken de accuracy en bewaar ze in accuracy_diabetes_svc
# ...

# Print de confusion matrix en het classification report ter controle
# ...

In [ ]:
# Train een KNeighborsClassifier met n_neighbors=5 op de trainingsdata
# ...

# Maak voorspellingen op de testdata en bewaar ze in y_pred_diabetes_knn
# ...

# Bereken de accuracy en bewaar ze in accuracy_diabetes_knn
# ...

# Vergelijk met de accuracy van het SVC-model: welk model presteert het beste?
# ...

## Oefening 2 — Customer Churn

De dataset `data/Telco-Customer-Churn.csv` bevat klantgegevens van een telecombedrijf; de kolom `Churn` geeft aan of een klant afhaakt (`Yes`/`No`). Dit is een **middelgrote dataset** (~7000 rijen) met zowel numerieke als categorische features, dus er is preprocessing nodig.

Zet een pipeline op met een `ColumnTransformer` die:
- numerieke features standaardiseert met een `StandardScaler`
- categorische features one-hot encodeert met een `OneHotEncoder(handle_unknown='ignore')`

Gebruik als classifier een `RandomForestClassifier(n_estimators=100, random_state=42)`. Splits met `test_size=0.2` en `random_state=42`. Bewaar de voorspellingen in `y_pred_churn` en de accuracy in `accuracy_churn`.

In [19]:
# Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Data inlezen
churn_data = pd.read_csv('data/Telco-Customer-Churn.csv')

# Features en labels
X = churn_data.drop(columns=['Churn'])
y = churn_data['Churn']

# Dataset onderzoeken
# print(churn_data.head())
# print(churn_data.tail())
# print(churn_data.info())
# print(y.value_counts(normalize=True)*100)
# Dit onderzoek leert mij dat de meeste features categorisch/binair zijn. Deze moeten gecodeerd worden. CustomerId heeft geen meerwaarde en mag verwijderd worden. De overige numerieke kolommen moeten geschaald worden. Daarnaast zijn de uitkomsten niet gebalanceerd. Er dient dus onderzocht te worden of balancering meerwaarde biedt. Er zijn nergens ontbrekende waarden, dus er hoeft niet geïmputeerd te worden.

# Splits met test_size=0.2 en random_state=42
X = X.drop(columns=['customerID'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bouw de preprocessor: StandardScaler voor numerieke features,
# OneHotEncoder(handle_unknown='ignore') voor categorische features
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Bouw de pipeline: preprocessor + RandomForestClassifier(n_estimators=100, random_state=42)
# en fit op de trainingsdata
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', oob_score=True))
])
param_grid_rf = {
    'model__max_depth': [None, 2, 5, 10, 20],
    'model__max_features': ['sqrt', 'log2']
}
grid_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf = grid_rf.fit(X_train, y_train)

# Maak voorspellingen op de testdata en bewaar ze in y_pred_churn
best_rf = grid_rf.best_estimator_
y_pred_churn = best_rf.predict(X_test)

# Bereken de accuracy en bewaar ze in accuracy_churn, print daarna
# confusion matrix en classification report
accuracy_churn = accuracy_score(y_test, y_pred_churn)
oob_churn = best_rf.named_steps['model'].oob_score_
print(oob_churn)
print(classification_report(y_test, y_pred_churn))
print(confusion_matrix(y_test, y_pred_churn))

# Het model scoort relatief goed met de optimale parameters. Hier dient vooral de afweging gemaakt te worden of de klassen gebalanceerd dienen te worden. Als men dit doet, krijgt men een evenwichtiger resultaat tussen het aantal vals positieven en negatieven. Als men dit niet doet, wordt het aantal vals positieven heel klein, maar het aantal vals negatieven heel groot. Van het aantal klanten die echt afhaken, worden er dus veel meer gemist. Balanceren lijkt mij hier dus beter.

0.7612708555200568
              precision    recall  f1-score   support

          No       0.88      0.82      0.85      1036
         Yes       0.59      0.69      0.63       373

    accuracy                           0.79      1409
   macro avg       0.73      0.76      0.74      1409
weighted avg       0.80      0.79      0.79      1409

[[853 183]
 [115 258]]


## Oefening 3 — Creditcardfraude

De dataset `data/creditcard.csv` bevat ~285.000 transacties waarvan de features al gepreprocessesd zijn (kolommen `V1` t/m `V28`, `Time`, `Amount`). De kolom `Class` is 1 bij fraude en 0 bij een valide transactie. Dit is een **grote dataset** én **sterk ongebalanceerd** (~500 fraudegevallen).

Train een `AdaBoostClassifier(n_estimators=50, random_state=42)` en een `XGBClassifier(n_estimators=50, random_state=42)` en vergelijk beide. Splits met `test_size=0.2` en `random_state=42`.

Bewaar de voorspellingen in `y_pred_creditcard_ada` / `y_pred_creditcard_xgb` en de accuracies in `accuracy_creditcard_ada` / `accuracy_creditcard_xgb`. Kijk zeker naar de confusion matrix / classification report, niet alleen naar de accuracy.

In [35]:
# Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Dataset inlezen
creditcard_data = pd.read_csv('data/creditcard.csv')

# Features en labels
X = creditcard_data.drop(columns=['Class'])
y = creditcard_data['Class']

# Dataset onderzoeken
# print(y.value_counts(normalize=True)*100)
# print(creditcard_data.head())
# print(creditcard_data.tail())
# print(creditcard_data.info())
# Het onderzoek leert dat zoals de opgave reeds aangaf, de dataset sterk ongebalanceerd is. Per defenitie zou AdaBoost hier moeilijker mee overweg kunnen, XGBoost lost dit onderliggend zelf op. Daarnaast zijn er geen ontbrekende waarden en dus geen imputatie nodig. Alle features zijn numeriek en dienen enkel geschaald te worden. Time heeft hier geen meerwaarde en mag gedropt worden.

# Splits met test_size=0.2 en random_state=42
X = X.drop(columns=['Time'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train AdaBoostClassifier(n_estimators=50, random_state=42)
# en bewaar de voorspellingen in y_pred_creditcard_ada, de accuracy in accuracy_creditcard_ada
pipeline_ab = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', AdaBoostClassifier(n_estimators=50, random_state=42))
])
param_grid_ab = {
    'model__learning_rate': [0.01,0.1,0.5,1,10,100]
}
grid_ab = GridSearchCV(pipeline_ab, param_grid_ab, cv=5, scoring='accuracy', n_jobs=-1)
grid_ab = grid_ab.fit(X_train, y_train)
best_ab = grid_ab.best_estimator_
y_pred_creditcard_ada = best_ab.predict(X_test)
accuracy_creditcard_ada = accuracy_score(y_test, y_pred_creditcard_ada)

# Train XGBClassifier(n_estimators=50, random_state=42)
# en bewaar de voorspellingen in y_pred_creditcard_xgb, de accuracy in accuracy_creditcard_xgb
pipeline_xg = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(n_estimators=50, random_state=42))
])
param_grid_xg = {
    'model__learning_rate': [0.01,0.1,0.5,1,10,100],
    'model__max_depth': [3, 5]
}
grid_xg = GridSearchCV(pipeline_xg, param_grid_xg, cv=5, scoring='accuracy', n_jobs=-1)
grid_xg = grid_xg.fit(X_train, y_train)
best_xg = grid_xg.best_estimator_
y_pred_creditcard_xgb = best_xg.predict(X_test)
accuracy_creditcard_xgb = accuracy_score(y_test, y_pred_creditcard_xgb)

# Vergelijk beide modellen met confusion matrix en classification report
print(classification_report(y_test, y_pred_creditcard_ada))
print(confusion_matrix(y_test, y_pred_creditcard_ada))
print(classification_report(y_test, y_pred_creditcard_xgb))
print(confusion_matrix(y_test, y_pred_creditcard_xgb))

# De resultaten tonen dat het XGBoost model een hogere recall heeft op de positieven. Dit model heeft minder vals negatieven, maar wel meer vals positieven. Vals negatieven zijn in dit geval echter een groter probleem. Men markeert liever iemand als verdacht die geen fraude pleegt dan omgekeerd. Adaboost presteert minder goed.

FileNotFoundError: [Errno 2] No such file or directory: 'data/creditcard.csv'

## Meerkeuzevragen

### Vraag 1
Je hebt een dataset met 500 rijen en 20 features. De business wil graag begrijpen **welke** factoren tot de voorspelling leiden. Welk model kies je bij voorkeur?

- **A)** Random Forest met 500 bomen
- **B)** Decision Tree
- **C)** XGBoost met 500 bomen
- **D)** Support Vector Machine

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_algoritme`.

### Vraag 2
De creditcard-dataset is sterk ongebalanceerd. Je model haalt 99.8% accuracy. Wat is de beste manier om te beoordelen of het model bruikbaar is voor het detecteren van fraude?

- **A)** Accuracy is het hoogst mogelijke getal, dus het model is uitstekend
- **B)** Kijk naar de confusion matrix / classification report, en vooral naar de detectie van de fraudeklasse
- **C)** Mean Squared Error berekenen
- **D)** R²-score berekenen

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_ongebalanceerd`.

In [26]:
# Vraag 1: antwoord_algoritme = "A", "B", "C" of "D"
antwoord_algoritme = "B"

# Vraag 2: antwoord_ongebalanceerd = "A", "B", "C" of "D"
antwoord_ongebalanceerd = "B"

In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Dataset inladen
hearts = pd.read_csv('data/heart_disease_uci.csv', sep=',', index_col=0)

# Dataset onderzoeken
# print(hearts.head())
# print(hearts.tail())
# print(hearts.info())
X = hearts.drop(columns=['num'])
y = hearts['num']
print(y.value_counts(normalize=True)*100)
# Het onderzoek toont dat de dataset klein is (minder dan 1000 records en minder dan 20 features). Dit betekent dat er verschillende mogelijheden zijn (KNN, SVC, Decision Tree, Random Forrest en AdaBoost en XGBoost). De boost-modellen lijken mij in dit geval overkill. SVC en een Decision Tree lijken mij een goede keuze, dankzij hun snelle classificatie en bij de Decision Tree het whitebox principe en het vermogen om met ontbrekende waarden om te gaan. Eventueel is een Randomforrest mogelijk als er bij de Decision Tree sprake is van overfitting. Er zijn zowel numerieke als categorische features met ontbrekende waarden. Schaling, imputatie en codering zijn dus noodzakelijk. De klassen van y zijn bovendien niet helemaal in balans, dus balancering kan eventueel toegepast worden.

# Data splitsen
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessor bouwen
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # Numerieke waarden worden met het gemiddelde geïmputeerd
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Categorische waarden worden met de meest voorkomende waarde geïmputeerd
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Pipelines modellen opzetten
pipeline_svc = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', SVC(random_state=42))
])
pipeline_dt = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])


# Optimalisatie met gridsearch
param_grid_svc = {
    'model__C': [0.1, 1, 5, 10, 100],
    'model__kernel': ['linear', 'rbf', 'poly']
}
param_grid_dt = {
    'model__max_depth': [2, 5, 10, 20, 50],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 5, 10, 20]
}
grid_svc = GridSearchCV(pipeline_svc, param_grid_svc, cv=5, scoring='accuracy', n_jobs=-1)
grid_dt = GridSearchCV(pipeline_dt, param_grid_dt, cv=5, scoring='accuracy', n_jobs=-1)

# Trainen en voorspellen
grid_svc = grid_svc.fit(X_train, y_train)
grid_dt = grid_dt.fit(X_train, y_train)
best_svc = grid_svc.best_estimator_
best_dt = grid_dt.best_estimator_
y_pred_svc = best_svc.predict(X_test)
y_pred_dt = best_dt.predict(X_test)
accuracy_svc = accuracy_score(y_test, y_pred_svc)
accuracy_dt = accuracy_score(y_test, y_pred_dt)

# Evaluatie
print(classification_report(y_test, y_pred_svc))
print(confusion_matrix(y_test, y_pred_svc))
print(classification_report(y_test, y_pred_dt))
print(confusion_matrix(y_test, y_pred_dt))
# Op basis van de evaluatiemetrieken scoren beide modellen niet goed en gelijkaardig. De decisiontree heeft echter een lager aantal vals negatieven, wat in medische context altijd beter is. Het balanceren van de modellen, helpt ook niet echt. Als ik meer tijd zou hebben, zou ik een Randomforrest proberen voor betere resultaten. Als dat niet werkt, zou ik pas over gaan naar XGBoost.

num
0    44.673913
1    28.804348
2    11.847826
3    11.630435
4     3.043478
Name: proportion, dtype: float64
              precision    recall  f1-score   support

           0       0.80      0.87      0.83        75
           1       0.59      0.35      0.44        54
           2       0.25      0.56      0.35        25
           3       0.22      0.08      0.11        26
           4       0.17      0.25      0.20         4

    accuracy                           0.55       184
   macro avg       0.41      0.42      0.39       184
weighted avg       0.57      0.55      0.54       184

[[65  6  4  0  0]
 [15 19 15  4  1]
 [ 1  4 14  3  3]
 [ 0  3 20  2  1]
 [ 0  0  3  0  1]]
              precision    recall  f1-score   support

           0       0.73      0.83      0.78        75
           1       0.44      0.61      0.51        54
           2       0.00      0.00      0.00        25
           3       0.21      0.19      0.20        26
           4       0.00      0.00    

/opt/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
